In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.artifacts_store import (
    apply_bot_cache,
    save_game_cache,
    train_or_load_bot_detector,
    try_load_game_cache,
)

N_PREVIEW = 5


def save_human_or_bot_cache(game, split, features_df, traces, session_ids, bot_types=None, user_ids=None):
    save_game_cache(
        game=game,
        split=split,
        features_df=features_df,
        traces=traces,
        session_ids=session_ids,
        bot_types=bot_types,
        user_ids=user_ids,
    )


## CSGO load: eye_vector → (dx, dy, time)

In [ ]:
from src.config import CSGO_DATA_ROOT
from src.data import (
    check_axis_convention,
    find_gameflt_files,
    gameflt_to_mouse,
    validate_real_roundtrip,
)

gameflt_paths = find_gameflt_files()
assert gameflt_paths, f"No gameFlt.csv under {CSGO_DATA_ROOT}"

first_path = gameflt_paths[0]
print(f"\n=== First-file checks: {first_path} ===")

mouse0, meta0, flt0 = gameflt_to_mouse(first_path)
print("convert meta:", meta0)
print(mouse0.head())
# check Y=pitch and angles round-trip
axis_ok = check_axis_convention(flt0["eyeVectorY"])
roundtrip = validate_real_roundtrip(
    flt0["eyeVectorX"], flt0["eyeVectorY"], flt0["eyeVectorZ"]
)

if not (axis_ok and roundtrip["ok"]):
    raise RuntimeError(
        "First-file checks failed — stop before converting all files. "
            f"axis_ok={axis_ok}, roundtrip_ok={roundtrip['ok']}"
        )

print("\nFirst file PASSED. Converting all gameFlt.csv ...")

csgo_mouse = {}
rows = []
for i, path in enumerate(gameflt_paths, 1):
    # path like .../S001/P3/gameFlt.csv
    participant = path.parent.name          # P3
    session = path.parent.parent.name       # S001
    key = (session, participant)
    try:
        mouse_df, meta, _ = gameflt_to_mouse(path)
    except Exception as e:
        print(f"  SKIP {key}: {e}")
        continue
    csgo_mouse[key] = mouse_df
    rows.append({
        "session": session,
        "participant": participant,
        "n_out": meta["n_out"],
        "teleport_frac": meta["teleport_frac"],
        "path": str(path),
    })
    if i % 50 == 0 or i == len(gameflt_paths):
        print(f"  converted {i}/{len(gameflt_paths)}")

csgo_convert_summary = pd.DataFrame(rows)
print(f"\nDone: {len(csgo_mouse)} traces")
print(csgo_convert_summary.head())
print(
    "n_out median:", csgo_convert_summary["n_out"].median(),
    "| teleport_frac median:", f"{csgo_convert_summary['teleport_frac'].median():.2%}",
)

## CSGO sessions & extract features


In [ ]:
from pathlib import Path

from src.features import extract_features
from src.config import CSGO_DATA_ROOT, CSGO_WINDOW_MIN
from src.data import window_mouse_round_alive

print(
    f"Extracting features from {len(csgo_mouse)} CSGO traces "
    f"(Round2+alive, {CSGO_WINDOW_MIN} min)"
)

csgo_cache = try_load_game_cache("csgo", "human")
csgo_mouse_win = {}
csgo_window_meta = []
if csgo_cache is not None:
    csgo_games_df = csgo_cache["features"]
    csgo_human_traces_cached = csgo_cache["traces"]
    for sid, trace in zip(csgo_cache["session_ids"], csgo_human_traces_cached):
        session, participant = sid.split("_", 1)
        csgo_mouse_win[(session, participant)] = trace
    print(f"Loaded CSGO human cache: features={len(csgo_games_df)} traces={len(csgo_human_traces_cached)}")
else:
    csgo_rows = []
    for (session, participant), mouse in csgo_mouse.items():
        session_dir = Path(CSGO_DATA_ROOT) / session / participant
        csgo_win, meta = window_mouse_round_alive(
            mouse, session_dir, window_min=CSGO_WINDOW_MIN
        )
        csgo_window_meta.append({"session": session, "participant": participant, **meta})

        feats = extract_features(csgo_win)

        csgo_mouse_win[(session, participant)] = csgo_win
        feats.update({
            "userId": f"csgo_{session}_{participant}",
            "gameId": f"{session}_{participant}",
            "session": session,
            "participant": participant,
            "is_bot": 0,
            "bot_type": "human",
            "round_n": meta["round_n"],
            "alive_frac_in_window": meta["alive_frac_in_window"],
        })
        csgo_rows.append(feats)

    csgo_games_df = pd.DataFrame(csgo_rows)
    csgo_traces = [csgo_mouse_win[(row.session, row.participant)] for _, row in csgo_games_df.iterrows()]
    save_human_or_bot_cache(
        game="csgo",
        split="human",
        features_df=csgo_games_df,
        traces=csgo_traces,
        session_ids=csgo_games_df["gameId"].astype(str).tolist(),
        bot_types=["human"] * len(csgo_traces),
        user_ids=csgo_games_df["userId"].astype(str).tolist(),
    )
    print(f"Saved CSGO human cache: features={len(csgo_games_df)} traces={len(csgo_traces)}")
csgo_window_meta_df = pd.DataFrame(csgo_window_meta)

print(f"Loaded {len(csgo_games_df)} CSGO human sessions ")
print(
    "alive_frac_in_window median:",
    f"{csgo_games_df['alive_frac_in_window'].median():.1%}",
)
print(csgo_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
re_cache = try_load_game_cache("re", "human")
if re_cache is not None:
    print(f"Red Eclipse — median n_events: {re_cache['features']['n_events'].median():.0f}")
lol_cache = try_load_game_cache("lol", "human")
if lol_cache is not None:
    print(f"LoL — median n_events: {lol_cache['features']['n_events'].median():.0f}")
print(f"CSGO — median n_events: {csgo_games_df['n_events'].median():.0f}")
print(csgo_games_df.head())


## CSGO trajectory preview


In [ ]:
from src.plotting import plot_trajectory
from src.config import CSGO_WINDOW_MIN

preview_keys = list(csgo_mouse_win.keys())[:5]

for session, participant in preview_keys:
    mouse = csgo_mouse_win[(session, participant)]
    plot_trajectory(
        mouse,
        title=f"CSGO {session}/{participant} (Round2+alive, {CSGO_WINDOW_MIN} min)",
    )
    print(f"\n{session}/{participant}, events={len(mouse)}")


## CSGO stitch bot generation


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_bot_rng = np.random.default_rng(RNG_SEED + 2)
csgo_segment_pool = []
for mouse in csgo_mouse_win.values():
    csgo_segment_pool.extend(build_segments(mouse, rng=csgo_bot_rng))

csgo_motion = collect_human_motion_samples(csgo_mouse_win.values(), rng=csgo_bot_rng)
csgo_target_ms = median_trace_duration_ms(csgo_mouse_win.values())
print(
    f"CSGO segment pool: {len(csgo_segment_pool)} segments from {len(csgo_mouse_win)} traces | "
    f"dt n={len(csgo_motion['dt_samples'])} sessions={len(csgo_motion['dt_by_session'])} median={np.median(csgo_motion['dt_samples']):.2f}ms | "
    f"step median={np.median(csgo_motion['step_samples']):.3f} | target={csgo_target_ms/1000:.1f}s"
)

csgo_bot_cache = try_load_game_cache('csgo', 'bot')
if csgo_bot_cache is not None:
    apply_bot_cache('csgo', csgo_bot_cache, globals(), n_preview=N_PREVIEW)
    CSGO_BOTS_READY = True
    print(f"Loaded CSGO bot cache: features={len(csgo_bot_cache['features'])} traces={len(csgo_bot_cache['traces'])}")
else:
    CSGO_BOTS_READY = False
    csgo_stitch_rows = []
    csgo_stitch_traces = []
    sample_csgo_stitch_trajectories = []
    for i in range(len(csgo_games_df)):
        bot_mouse = stitch_bot_game(
            csgo_segment_pool,
            dt_samples=csgo_motion['dt_samples'],
            dt_by_session=csgo_motion['dt_by_session'],
            target_duration_ms=csgo_target_ms,
            rng=csgo_bot_rng,
        )
        csgo_stitch_traces.append(bot_mouse)
        if len(sample_csgo_stitch_trajectories) < N_PREVIEW:
            sample_csgo_stitch_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -1, 'gameId': f'csgo_stitch_{i}', 'is_bot': 1, 'bot_type': 'stitch'})
        csgo_stitch_rows.append(feats)
    csgo_bots_stitch_df = pd.DataFrame(csgo_stitch_rows)
    print(f"CSGO stitch bots: {len(csgo_bots_stitch_df)}")
    print(csgo_bots_stitch_df.head())


## CSGO stitch bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_stitch_trajectories):
    print(f"csgo_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"csgo_stitch_{i}")


## CSGO smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_generator_params,
    smooth_params_for_print,
)
from src.features import extract_features

csgo_median_events = int(csgo_games_df['n_events'].median())
csgo_smooth_params = estimate_smooth_params(csgo_games_df, **csgo_motion)
print(f"CSGO smooth params: {smooth_params_for_print(csgo_smooth_params)}")

if not CSGO_BOTS_READY:
    csgo_smooth_gen = smooth_generator_params(csgo_smooth_params)
    csgo_smooth_rows = []
    csgo_smooth_traces = []
    sample_csgo_smooth_trajectories = []
    for i in range(len(csgo_games_df)):
        bot_mouse = generate_smooth_bot_game(
            n_events=max(csgo_median_events * 3, 1),
            seed=RNG_SEED + 220 + i,
            target_duration_ms=csgo_target_ms,
            **csgo_smooth_gen,
        )
        csgo_smooth_traces.append(bot_mouse)
        if len(sample_csgo_smooth_trajectories) < N_PREVIEW:
            sample_csgo_smooth_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -2, 'gameId': f'csgo_smooth_{i}', 'is_bot': 1, 'bot_type': 'smooth'})
        csgo_smooth_rows.append(feats)
    csgo_bots_smooth_df = pd.DataFrame(csgo_smooth_rows)
    print(f"CSGO smooth bots: {len(csgo_bots_smooth_df)}")
else:
    print('CSGO smooth bots loaded from cache')


## CSGO smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_smooth_trajectories):
    plot_trajectory(df, title=f"csgo_smooth_{i}")
    print(f"csgo_smooth_{i}, events={len(df)}")


## CSGO Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features

csgo_median_events = int(csgo_games_df['n_events'].median())
csgo_bezier_params = estimate_bezier_params(csgo_games_df, **csgo_motion)
print(f"CSGO bezier params: {bezier_params_for_print(csgo_bezier_params)}")

if not CSGO_BOTS_READY:
    csgo_bezier_rows = []
    csgo_bezier_traces = []
    sample_csgo_bezier_trajectories = []
    for i in range(len(csgo_games_df)):
        bot_mouse = generate_bezier_bot_game(
            n_events=max(csgo_median_events * 3, 1),
            seed=RNG_SEED + 250 + i,
            target_duration_ms=csgo_target_ms,
            **csgo_bezier_params,
        )
        csgo_bezier_traces.append(bot_mouse)
        if len(sample_csgo_bezier_trajectories) < N_PREVIEW:
            sample_csgo_bezier_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -3, 'gameId': f'csgo_bezier_{i}', 'is_bot': 1, 'bot_type': 'bezier'})
        csgo_bezier_rows.append(feats)
    csgo_bots_bezier_df = pd.DataFrame(csgo_bezier_rows)
    print(f"CSGO bezier bots: {len(csgo_bots_bezier_df)}")
else:
    print('CSGO bezier bots loaded from cache')


## CSGO Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_bezier_trajectories):
    plot_trajectory(df, title=f"csgo_bezier_{i}")
    print(f"csgo_bezier_{i}, events={len(df)}")



## CSGO VAE bot — train once / load weights


In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_CSGO_WEIGHTS

CSGO_VAE_FORCE_RETRAIN = False
CSGO_VAE_WEIGHTS_PATH = DEFAULT_CSGO_WEIGHTS

csgo_step_median = float(np.median(csgo_motion["step_samples"]))
csgo_vae_bundle = ensure_vae_bundle(
    list(csgo_mouse_win.values()),
    csgo_step_median,
    path=CSGO_VAE_WEIGHTS_PATH,
    force_retrain=CSGO_VAE_FORCE_RETRAIN,
    seed=RNG_SEED + 2,
)
print(
    f"CSGO VAE ready | path={Path(CSGO_VAE_WEIGHTS_PATH)} | "
    f"seg_len={csgo_vae_bundle['seg_len']} z={csgo_vae_bundle['z_dim']} "
    f"norm={csgo_vae_bundle.get('norm')} axis_scale={csgo_vae_bundle.get('axis_scale')} "
    f"trained_segments={csgo_vae_bundle.get('n_segments')}"
)


## CSGO VAE bot generation


In [ ]:
from src.vae_bot import generate_vae_bot_games
from src.features import extract_features
from src.config import RNG_SEED, VAE_POOL_SEGMENTS

if not CSGO_BOTS_READY:
    csgo_vae_rng = np.random.default_rng(RNG_SEED + 5)
    csgo_vae_traces = generate_vae_bot_games(
        csgo_vae_bundle,
        n_games=len(csgo_games_df),
        dt_samples=csgo_motion['dt_samples'],
        dt_by_session=csgo_motion['dt_by_session'],
        target_duration_ms=csgo_target_ms,
        n_pool_segments=VAE_POOL_SEGMENTS,
        rng=csgo_vae_rng,
    )
    sample_csgo_vae_trajectories = [m.copy() for m in csgo_vae_traces[:N_PREVIEW]]
    csgo_vae_rows = []
    for i, bot_mouse in enumerate(csgo_vae_traces):
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -4, 'gameId': f'csgo_vae_{i}', 'is_bot': 1, 'bot_type': 'vae'})
        csgo_vae_rows.append(feats)
    csgo_bots_vae_df = pd.DataFrame(csgo_vae_rows)
    print(f"CSGO VAE bots: {len(csgo_bots_vae_df)}")
    print(csgo_bots_vae_df.head())
else:
    print('CSGO vae bots loaded from cache')


In [ ]:
if not globals().get('CSGO_BOTS_READY'):
    csgo_bot_features = pd.concat(
        [csgo_bots_stitch_df, csgo_bots_smooth_df, csgo_bots_bezier_df, csgo_bots_vae_df], ignore_index=True
    )
    csgo_bot_traces = csgo_stitch_traces + csgo_smooth_traces + csgo_bezier_traces + csgo_vae_traces
    save_human_or_bot_cache(
        game='csgo',
        split='bot',
        features_df=csgo_bot_features,
        traces=csgo_bot_traces,
        session_ids=csgo_bot_features['gameId'].astype(str).tolist(),
        bot_types=csgo_bot_features['bot_type'].astype(str).tolist(),
        user_ids=csgo_bot_features['userId'].astype(str).tolist(),
    )
    CSGO_BOTS_READY = True
    print(f"Saved CSGO bot cache: features={len(csgo_bot_features)} traces={len(csgo_bot_traces)}")


## CSGO VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_vae_trajectories):
    plot_trajectory(df, title=f"csgo_vae_{i}")
    print(f"\ncsgo_vae_{i}, events={len(df)}")


## CSGO in-domain (GroupKFold + split-player + **windows**)

In [ ]:
from src.evaluation import evaluate_group_kfold_windows
from src.features import feature_cols
from src.config import RNG_SEED

def _csgo_traces_for(feat_df):
    return [
        csgo_mouse_win[(row.session, row.participant)]
        for row in feat_df.itertuples(index=False)
    ]

csgo_cv = evaluate_group_kfold_windows(
    human_df=csgo_games_df,
    groups=csgo_games_df["participant"].to_numpy(),
    traces_for_df=_csgo_traces_for,
    feature_cols=feature_cols,
    rng_seed_base=RNG_SEED + 2,
    round_deltas=False,
    vae_bundle=csgo_vae_bundle,
    name="CSGO",
)
csgo_cv_summary = csgo_cv["summary_df"]
csgo_cv_session_summary = csgo_cv["session_summary_df"]
csgo_cv_folds = csgo_cv["fold_df"]
print("\nCSGO fold table (window counts):")
print(
    csgo_cv_folds[
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nCSGO window-level summary:")
print(csgo_cv_summary.to_string(index=False))
if csgo_cv_session_summary is not None:
    print("\nCSGO session-mean-of-windows summary:")
    print(csgo_cv_session_summary.to_string(index=False))


## Cross-game transfer (CSGO train → RE test)

In [ ]:
from src.features import cross_game_feature_cols

print("=== CSGO model trained on STITCH bots (7 cross-game features) ===")
csgo_x_model_stitch, csgo_x_acc_stitch = train_or_load_bot_detector(
    csgo_games_df, csgo_bots_stitch_df, cross_game_feature_cols,
    train_game="csgo", feature_set="raw", bot_type="stitch",
    name="CSGO stitch",
)
print()
print("=== CSGO model trained on SMOOTH bots (7 cross-game features) ===")
csgo_x_model_smooth, csgo_x_acc_smooth = train_or_load_bot_detector(
    csgo_games_df, csgo_bots_smooth_df, cross_game_feature_cols,
    train_game="csgo", feature_set="raw", bot_type="smooth",
    name="CSGO smooth",
)
print()
print("=== CSGO model trained on BEZIER bots (7 cross-game features) ===")
csgo_x_model_bezier, csgo_x_acc_bezier = train_or_load_bot_detector(
    csgo_games_df, csgo_bots_bezier_df, cross_game_feature_cols,
    train_game="csgo", feature_set="raw", bot_type="bezier",
    name="CSGO bezier",
)

print()
print("=== CSGO model trained on VAE bots (7 cross-game features) ===")
csgo_x_model_vae, csgo_x_acc_vae = train_or_load_bot_detector(
    csgo_games_df, csgo_bots_vae_df, cross_game_feature_cols,
    train_game="csgo", feature_set="raw", bot_type="vae",
    name="CSGO vae",
)


## Scale-invariant train (CSGO) — for RE transfer



In [ ]:
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_si_stitch_tr = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth_tr = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier_tr = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae_tr = to_scale_invariant(csgo_bots_vae_df)

print("=== Train on CSGO (scale-invariant features) ===")
m_csgo_si_stitch, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_stitch_tr, SCALE_INVARIANT_COLS,
    train_game="csgo", feature_set="si_min", bot_type="stitch",
    random_state=RNG_SEED, name="CSGO scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_csgo_si_smooth, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_smooth_tr, SCALE_INVARIANT_COLS,
    train_game="csgo", feature_set="si_min", bot_type="smooth",
    random_state=RNG_SEED, name="CSGO scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_csgo_si_bezier, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_bezier_tr, SCALE_INVARIANT_COLS,
    train_game="csgo", feature_set="si_min", bot_type="bezier",
    random_state=RNG_SEED, name="CSGO scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_csgo_si_vae, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_vae_tr, SCALE_INVARIANT_COLS,
    train_game="csgo", feature_set="si_min", bot_type="vae",
    random_state=RNG_SEED, name="CSGO scale-invariant vae",
    show_feature_importance=False,
)


## Scale-invariant EXT train (CSGO, 10 feats) — for RE transfer


In [ ]:
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_si_stitch_tr = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth_tr = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier_tr = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae_tr = to_scale_invariant(csgo_bots_vae_df)

print("=== Train on CSGO (scale-invariant EXT, 10 feats) ===")
print("cols:", SCALE_INVARIANT_EXT_COLS)
m_csgo_si_ext_stitch, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_stitch_tr, SCALE_INVARIANT_EXT_COLS,
    train_game="csgo", feature_set="si_ext", bot_type="stitch",
    random_state=RNG_SEED, name="CSGO SI-EXT stitch",
    show_feature_importance=False,
)
print()
m_csgo_si_ext_smooth, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_smooth_tr, SCALE_INVARIANT_EXT_COLS,
    train_game="csgo", feature_set="si_ext", bot_type="smooth",
    random_state=RNG_SEED, name="CSGO SI-EXT smooth",
    show_feature_importance=False,
)
print()
m_csgo_si_ext_bezier, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_bezier_tr, SCALE_INVARIANT_EXT_COLS,
    train_game="csgo", feature_set="si_ext", bot_type="bezier",
    random_state=RNG_SEED, name="CSGO SI-EXT bezier",
    show_feature_importance=False,
)
print()
m_csgo_si_ext_vae, _ = train_or_load_bot_detector(
    csgo_si_human_tr, csgo_si_vae_tr, SCALE_INVARIANT_EXT_COLS,
    train_game="csgo", feature_set="si_ext", bot_type="vae",
    random_state=RNG_SEED, name="CSGO SI-EXT vae",
    show_feature_importance=False,
)
